In [2]:
%pip install -U -r requirements.txt

  Using cached aiohttp-3.11.11-cp310-cp310-macosx_10_9_x86_64.whl.metadata (7.7 kB)
Note: you may need to restart the kernel to use updated packages.


In [38]:
from pipeline import Pipeline

pipeline = Pipeline(lookback_periods=30, forward_periods=7)


2025-01-06 08:45:32 DEBUG pipeline Creating Spark session...
2025-01-06 08:45:32 DEBUG pipeline Spark session created.


In [39]:
reload = False

timeframe = "1d"
candles_file_name = f"ohlcv{timeframe}"
candles_path = f"{pipeline.data_dir}/{candles_file_name}.csv"

if reload:
    candles_df = await pipeline.get_candles_df(timeframe=timeframe)  # noqa: PLE1142
    pipeline.save_csv(candles_file_name, candles_df)

candles_df = pipeline.spark.read.csv(candles_path, header=True, inferSchema=True)

candles_df.show()

+-------------------+--------+--------+--------+--------+-------------+---------------+------+
|          timestamp|    open|    high|     low|   close|       volume|         symbol|ticker|
+-------------------+--------+--------+--------+--------+-------------+---------------+------+
|2024-01-01 14:00:00| 42327.0| 44265.0| 42225.0| 44241.0|    699.16021|  BTC/USDC:USDC|   BTC|
|2024-01-01 14:00:00|  2284.3|  2355.5|  2268.0|  2355.5|   27866.0026|  ETH/USDC:USDC|   ETH|
|2024-01-01 14:00:00|    10.6|  11.218|  10.464|  11.214|     117550.0| ATOM/USDC:USDC|  ATOM|
|2024-01-01 14:00:00| 0.97254|  1.0201| 0.95815|  1.0175|    4032700.6|MATIC/USDC:USDC| MATIC|
|2024-01-01 14:00:00|   2.952|  3.0809|  2.8993|  3.0746|     139655.1| DYDX/USDC:USDC|  DYDX|
|2024-01-01 14:00:00|  101.86|  110.17|   101.6|  110.17|    208581.02|  SOL/USDC:USDC|   SOL|
|2024-01-01 14:00:00|  38.584|  42.083|  38.086|  41.935|    109588.87| AVAX/USDC:USDC|  AVAX|
|2024-01-01 14:00:00|  311.33|  315.52|  306.97|  

In [40]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import window as W

# Get oldest and latest timestamps
oldest = candles_df.select(F.min("timestamp")).first()[0]
latest = candles_df.select(F.max("timestamp")).first()[0]
pipeline.logger.info("Data range: %s to %s", oldest, latest)

symbol_window = W.Window.partitionBy("symbol").orderBy("timestamp")
window_size = 17 * 24
rolling_window = symbol_window.rowsBetween(-window_size + 1, 0)


def with_forward_return(df: DataFrame) -> DataFrame:
    forward_window = 7
    forward_window = symbol_window.rowsBetween(1, forward_window)
    return df.withColumn(
        "forward_return", F.exp(F.sum("return").over(forward_window)) - 1
    ).withColumn(
        "price_zscore_fw_return_corr",
        F.corr(F.col("price_zscore"), F.col("forward_return")).over(rolling_window),
    )

return_df = (
    candles_df
    .transform(pipeline.with_returns)
    .transform(pipeline.with_bollinger)
    .transform(pipeline.with_zscore)
    # .transform(pipeline.with_auto_regression)
    # .transform(with_forward_return)
    .transform(pipeline.with_volatility)
    .transform(pipeline.with_beta)
)

return_df.describe().show()
return_df.printSchema()

2025-01-06 08:45:37 INFO pipeline Data range: 2024-01-01 14:00:00 to 2025-01-04 14:00:00
2025-01-06 08:45:37 DEBUG pipeline Calculating returns...
2025-01-06 08:45:37 DEBUG pipeline Calculating bollinger bands...


+-------------------+------+------+------+------+--------+--------------+------+-----+--------------------+--------------------+----------+
|          timestamp|  open|  high|   low| close|  volume|        symbol|ticker|count|              return|          log_return|cum_return|
+-------------------+------+------+------+------+--------+--------------+------+-----+--------------------+--------------------+----------+
|2024-01-01 14:00:00|108.72| 116.4|107.81|116.32|10422.87|AAVE/USDC:USDC|  AAVE|    1|                NULL|                NULL|      NULL|
|2024-01-02 14:00:00|116.37|120.03|109.34|110.38|16823.79|AAVE/USDC:USDC|  AAVE|    2|-0.05106602475928471|-0.05241605577385...|      NULL|
|2024-01-03 14:00:00| 110.4|113.16| 91.63| 102.6|41874.86|AAVE/USDC:USDC|  AAVE|    3|-0.07048378329407502|-0.07309102527468606|      NULL|
|2024-01-04 14:00:00|102.66|105.39|101.35|103.82|10124.57|AAVE/USDC:USDC|  AAVE|    4| 0.01189083820662767|0.011820697662413743|      NULL|
|2024-01-05 14:00:00

2025-01-06 08:45:41 DEBUG pipeline Calculating beta...


+-------+-----------------+-----------------+-----------------+-----------------+-------------------+---------------+------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+-----------------+------------------+--------------------+-------------------+--------------------+--------------------+---------------------+--------------------+------------------+--------------------+--------------------+-------------------+-------------------+
|summary|             open|             high|              low|            close|             volume|         symbol|ticker|              return|          log_return|         cum_return|                 sma|         mean_return|        price_stddev|       return_stddev|     bollinger_upper|   bollinger_lower|              max|               min|        price_zscore|              z_max|            z_to_max|              st

In [41]:
# Get latest entries
latest_entries = (
    return_df.filter(F.col("timestamp") == F.lit(latest))
    .orderBy("symbol")
    .drop("forward_return", "price_zscore_fw_return_corr")
    .dropna()
)

latest_entries.show(vertical=True)
latest_entries.count()


-RECORD 0-------------------------------------
 timestamp             | 2025-01-04 14:00:00  
 open                  | 351.15               
 high                  | 351.55               
 low                   | 341.18               
 close                 | 342.81               
 volume                | 8309.4               
 symbol                | AAVE/USDC:USDC       
 ticker                | AAVE                 
 return                | -0.02377833466226228 
 log_return            | -0.02406560223127... 
 cum_return            | 0.38185262818445587  
 sma                   | 332.4253333333333    
 mean_return           | 0.010780836101151266 
 price_stddev          | 34.63804851687621    
 return_stddev         | 0.07551907121044554  
 bollinger_upper       | 401.7014303670857    
 bollinger_lower       | 263.1492362995809    
 max                   | 400.48               
 min                   | 218.66               
 price_zscore          | 0.29980518855174904  
 z_max       

148

In [50]:
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql import types as T


@F.udf(returnType=T.FloatType())
def long_score(  # noqa: PLR0913
    # ticker: str,
    close: float,
    min_price: float,
    max_price: float,
    mean_return: float,
    return_stddev: float,
    beta: float,
    price_zscore: float,
) -> float:
    drawdown_factor = -((close / max_price) - 1)
    ranup_factor = (close / min_price) - 1
    range_factor = drawdown_factor / ranup_factor

    annualized_return = mean_return * 365
    annualized_volatility = return_stddev * np.sqrt(365)
    risk_free_rate = 0.045
    risk_adjusted_return_factor = (annualized_return - risk_free_rate) / (
        annualized_volatility * beta
    )

    mean_reversion_factor = np.exp(-price_zscore * 0.01)

    return float(range_factor * risk_adjusted_return_factor * mean_reversion_factor)


long_score_df = latest_entries.withColumn(
    "long_score",
    long_score(
        # F.col("ticker"),
        F.col("close"),
        F.col("min"),
        F.col("max"),
        F.col("mean_return"),
        F.col("return_stddev"),
        F.col("beta"),
        F.col("price_zscore"),
    ),
).orderBy("long_score", ascending=False)

long_picks = long_score_df.select("symbol", "beta", "price_stddev", "long_score").limit(30)


AAVE 0.25362481252863056 1.9776066704493513 0.9970064377841616 0.5000686396851229
AI 0.17483867648139628 -0.19477578104978016 0.9884518122609042 -0.0336610738603861
ALGO 0.3879579248717981 -0.26995554059390986 0.9942463091999959 -0.10412879929365351
ALT 1.7487703321973556 -1.9663156514203481 1.0045325834014571 -3.4542203724840093
ATOM 1.1038543437539348 -1.2650738097717036 1.0022830621828769 -1.3996454187550174
AVAX 0.94602778831755 -0.9459778920791025 1.00175019408175 -0.8964876591316191
BANANA 1.30235461897954 -2.986233956605998 1.0021063774963788 -3.8973275744196423
BIGTIME 2.5949446679399557 -2.3186329156443692 1.0046890229083187 -6.044936678599527
BLUR 2.296417501781239 -2.511342410060247 1.005540743274272 -5.7990446322333336
BNT 1.1238175413378377 -1.5538729146302723 1.0037084038547912 -1.752745511530064
BOME 1.3483063040748893 -1.8157809481916884 1.0029769201528305 -2.4555170812149
BSV 2.012414123005455 -2.8204856600932615 1.0050694914083986 -5.70475953419051
CAKE 0.876912158039

+----------------+------------------+--------------------+-------------+
|          symbol|              beta|        price_stddev|   long_score|
+----------------+------------------+--------------------+-------------+
|  HYPE/USDC:USDC| 1.218835949167042|  5.5697186091451245|    1.0404797|
|   SCR/USDC:USDC| 1.568885338967108| 0.09420089052218363|    0.6206715|
|  AAVE/USDC:USDC|1.3633494545898432|   34.63804851687621|   0.50006866|
|   ZEN/USDC:USDC|1.6587570802742597|   8.511790799596286|   0.44208807|
| SUSHI/USDC:USDC|3.3134804943748453| 0.34925373222238354|    0.3164434|
| TURBO/USDC:USDC|  2.76561158490456|0.001274259841507...|   0.26249817|
|   BTC/USDC:USDC|0.9999999999999998|  3516.4022675024744|   0.25765705|
|  PURR/USDC:USDC|1.8943641510849716| 0.10173620314492196|   0.25022376|
|    IO/USDC:USDC|1.6226589850420419| 0.45437616220661176|   0.24372518|
|   XRP/USDC:USDC|1.5812419219596556| 0.15004911230476606|    0.2173891|
|   ENA/USDC:USDC|1.7409569566819902| 0.09626522126

ACE 1.6112486041227707 -1.8046694876142522 1.0048682675385148 -2.9219270009288607
ADA 0.2597006011772159 -0.38810341730877934 0.9913546744401457 -0.09991932245870497
APE 0.8328490252906321 -1.4903637683277793 1.0013886807760155 -1.2429717090324912
APT 2.302759927122036 -1.8467619255888834 1.0066083208118117 -4.2807522284332045
AR 1.0778236017816738 -1.998109093421316 1.0021912204296792 -2.158328172168804
ARB 1.306486976556898 -1.7775477475869739 1.0033297904406373 -2.3300758978932077
ARK 1.1819400773727593 -2.1564337080067353 1.0036245117058402 -2.558013490049432
BADGER 0.16428776640036852 -0.02740961763656546 0.9921820977335358 -0.004467860338429153
BCH 1.225945919379981 -2.119854468795801 1.003895407655674 -2.608950426040726
BLAST 1.2717462545858944 -1.1385900022967819 1.004797056612436 -1.4549436972522978
BNB 0.312018538034171 0.04169028929361782 0.9919040719351555 0.012902830124695053
BRETT 0.8905338344126585 -1.2548272244816563 1.0009069045155654 -1.1184795347948915
BTC 1.32713017

In [51]:

@F.udf(returnType=T.FloatType())
def short_score(  # noqa: PLR0913
    # ticker: str,
    close: float,
    min_price: float,
    max_price: float,
    mean_return: float,
    return_stddev: float,
    beta: float,
    price_zscore: float,
) -> float:
    smoked_factor = (close / min_price) - 1
    ranup_factor = 1 - (close / max_price)
    range_factor = smoked_factor / ranup_factor

    annualized_return = mean_return * 365
    annualized_volatility = return_stddev * np.sqrt(365)
    risk_free_rate = 0.045
    risk_adjusted_return_factor = (- annualized_return - risk_free_rate) / (
        annualized_volatility * beta
    )

    mean_reversion_factor = np.exp(price_zscore * 0.01)

    return float(range_factor * risk_adjusted_return_factor * mean_reversion_factor)


short_score_df = latest_entries.withColumn(
    "short_score",
    short_score(
        # F.col("ticker"),
        F.col("close"),
        F.col("min"),
        F.col("max"),
        F.col("mean_return"),
        F.col("return_stddev"),
        F.col("beta"),
        F.col("price_zscore"),
    ),
).orderBy("short_score", ascending=False)

short_picks = short_score_df.select("symbol", "beta", "price_stddev", "short_score").limit(30)

AAVE 3.9428318942063862 -2.023361005532616 1.003002550537679 -8.001725970634686]
AI 5.719558281524769 0.17127836467286106 1.0116831064457068 0.991081787659371
ALGO 2.577599105187639 0.23690685122789729 1.0057869873357979 0.6141847166917818
ALT 0.5718303779453333 1.9287892599339465 0.9954878682121895 1.097963679535662
ATOM 0.9059166235639823 1.2292983278110086 0.9977221383169893 1.1111050685202568
AVAX 1.0570514020295707 0.9077635648159657 0.9982528637457824 0.95787627950464
BANANA 0.7678400225458946 2.9092001419697597 0.9978980500037917 2.229104966062999
BIGTIME 0.3853646716844519 2.276934424109359 0.9953328614114394 0.8733549056343178
BLUR 0.4354608860210916 2.467538257713101 0.994489787399136 1.0685955822096427
BNT 0.8898241602543057 1.5011098921227342 0.9963052975938541 1.3307887470879298
BOME 0.7416712337380399 1.7765166063247448 0.9970319155974428 1.3136805410918317
BSV 0.4969156142208655 2.7641562340542976 0.9949560787072596 1.3666243026865474
CAKE 1.1403650762873028 1.4184186146

+------------------+------------------+--------------------+-----------+
|            symbol|              beta|        price_stddev|short_score|
+------------------+------------------+--------------------+-----------+
|     GMT/USDC:USDC|0.8253419350007895|0.033700609524144394|  2.6419423|
|     TON/USDC:USDC|1.2078320527053696|   0.443531973422746|  2.4404337|
|   LISTA/USDC:USDC|1.1842822571302378| 0.06584229848311969|    2.38355|
|  BANANA/USDC:USDC|1.2579666033440209|   7.547897233552928|   2.229105|
|     TRX/USDC:USDC|1.1377182665549364|0.022654433148158154|  2.1771324|
|CHILLGUY/USDC:USDC| 2.148429678456518| 0.09373356895949848|  2.1062887|
|     TAO/USDC:USDC|1.5892352708709905|    77.0949054804173|   2.095548|
|    MERL/USDC:USDC| 1.474877513333683|0.059836146884830355|   2.016147|
|  RENDER/USDC:USDC|1.6697675179913543|  1.1542613684621603|  1.9838359|
|     GAS/USDC:USDC|1.5827956185799221|  0.7205918124995446|  1.9752613|
|     OGN/USDC:USDC|1.5868632343538076|0.0163762857

ACE 0.6206366897332026 1.772180924243844 0.9951553176712009 1.0945519307971088
ADA 3.8505879288189044 0.34117488525545725 1.0087207190148537 1.3251805117303215
APE 1.2006978091269767 1.454253150160959 0.9986132449840163 1.7436971326269952
APT 0.4342615086453189 1.8066399062487908 0.9934350624019458 0.7794036220897802
AR 0.9277956043521137 1.9565522001821352 0.997813570519267 1.8113115481456314
ARB 0.7654113802461234 1.7311574812066803 0.9966812602671999 1.3206501488726652
ARK 0.8460665808226256 2.1107726606706576 0.9963885779357065 1.7794047346174509
BADGER 6.0868804897073385 -0.012959458656114916 1.0078795034543788 -0.07950423236950986
BCH 0.8156966667059403 2.049483929633151 0.9961197076647946 1.6652703031813736
BLAST 0.7863203814393144 1.1054475339498526 0.9952258452780417 0.8650860597533315
BNB 3.204937778057546 -0.2498781082596144 1.0081620070870863 -0.8073802817528364
BRETT 1.1229219613643748 1.222832989701102 0.9990939172150037 1.3719018352469259
BTC 0.7535055886045189 -0.400107

In [52]:
picks = long_picks.withColumn("type", F.lit("long")).union(
  short_picks.withColumn("type", F.lit("short")),
);

picks.show();

AAVE 0.25362481252863056 1.9776066704493513 0.9970064377841616 0.5000686396851229
AI 0.17483867648139628 -0.19477578104978016 0.9884518122609042 -0.0336610738603861
ALGO 0.3879579248717981 -0.26995554059390986 0.9942463091999959 -0.10412879929365351
ALT 1.7487703321973556 -1.9663156514203481 1.0045325834014571 -3.4542203724840093
ATOM 1.1038543437539348 -1.2650738097717036 1.0022830621828769 -1.3996454187550174
AVAX 0.94602778831755 -0.9459778920791025 1.00175019408175 -0.8964876591316191
BANANA 1.30235461897954 -2.986233956605998 1.0021063774963788 -3.8973275744196423
BIGTIME 2.5949446679399557 -2.3186329156443692 1.0046890229083187 -6.044936678599527
BLUR 2.296417501781239 -2.511342410060247 1.005540743274272 -5.7990446322333336
BNT 1.1238175413378377 -1.5538729146302723 1.0037084038547912 -1.752745511530064
BOME 1.3483063040748893 -1.8157809481916884 1.0029769201528305 -2.4555170812149
BSV 2.012414123005455 -2.8204856600932615 1.0050694914083986 -5.70475953419051
CAKE 0.876912158039